# 5장 도구

에이전트가 도구를 호출해 외부 정보를 가져오고 작업을 수행합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 5장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

---

## 1. 언어 모델 선택

Gemma 3는 내장 도구 호출을 지원하지 않습니다. 이번 실습에서는 프롬프트로 도구 이름과 인자를 JSON으로 출력하게 합니다. 이 출력을 해석할 때 어떤 문제가 생길 수 있는지도 살펴봅니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B(내장 추론과 도구 호출 없음)
llm = LLM(model="gemma3:12b")

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-3-12b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-3-12b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. `Tools` 추가

앞 장에서 `TinyAgent`에 메모리를 연결했습니다. 이제 외부 함수를 호출하는 기능을 추가합니다.

![에이전트에 도구 모듈을 연결하는 구조](../images/ch5.png)

도구를 사용하는 방법에는 함수 호출, JSON 형식 지정, MCP, 스킬 등이 있습니다. 특정 도구 호출 형식으로 학습하지 않은 모델이라도 프롬프트의 지시를 따르면 호출할 함수와 인자를 출력할 수 있습니다. 이번 실습은 다음 순서로 진행합니다.

1. 도구로 쓸 함수 작성
2. 함수의 용도와 입력 형식 정의
3. 사용할 도구 선택
4. 함수 호출
5. 도구의 반환값 처리

![도구 작성부터 결과 처리까지의 순서](../images/ch5_tools.png)

## 3. 도구 함수 작성

도구를 등록하기 전에 먼저 함수를 만듭니다. 여기서는 인자를 받아 문자열을 반환하는 함수 두 개를 사용합니다. 하나는 두 수를 곱하고, `get_weather`는 임의로 정한 기온을 반환합니다.

In [ ]:
import random

def multiply(a: str, b: str) -> float:
    return float(a) * float(b)

def get_weather(location: str) -> str:
    temperature = random.randint(10, 30)
    status = random.choice(["Sunny", "Cloudy", "Rainy"])
    return f"Weather in {location}: {status}, {temperature}℃"

각 함수가 하나의 도구입니다. 함수가 복잡할수록 모델이 용도와 인자의 형식을 이해할 수 있게 설명해야 합니다. 파이썬에서는 `multiply(a, b)`를 바로 호출할 수 있지만 모델의 출력은 텍스트입니다. 따라서 호출 프로그램이 도구 이름을 실제 함수에 연결해 실행합니다. 외부 API도 이 방식으로 호출할 수 있습니다.

아래 `Tools` 클래스는 `registry`라는 딕셔너리에 도구 이름과 함수를 짝지어 저장합니다.

In [ ]:
from typing import Callable

class Tools:
    """에이전트가 사용할 도구를 등록하고 관리한다."""

    def __init__(self, requires_approval: list[str] = []):
        """도구 목록을 초기화하고 실행 전 승인이 필요한 도구를 지정한다.
        """
        self.registry = {}
        self.requires_approval = requires_approval

    def add_tool(self, name: str, func: Callable, description: str):
        """에이전트가 사용할 도구를 등록한다.

        매개변수:
            name: 도구의 이름.
            func: 도구의 동작을 구현한 함수.
            description: 도구의 용도.
        """
        self.registry[name] = {"function": func, "description": description}

    @property
    def schemas(self):
        """모델 자체의 도구 호출에서만 사용한다."""
        return None

`add_tool`로 도구를 등록한 뒤 이름으로 함수를 참조합니다.

In [ ]:
# 도구 등록
tools = Tools()
tools.add_tool("multiply", multiply, "Multiplies two numbers: multiply(a: str, b: str)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(location: str)")

## 4. 모델에 전달할 도구 설명

주로 함수의 문서 문자열(docstring)로 용도를 설명합니다. 여기서는 도구마다 짧은 설명을 직접 붙이고 `Tools`가 등록과 사용을 관리하게 합니다.

- `add_tool`은 에이전트가 사용할 도구를 추가합니다.
- `descriptions`는 등록한 도구의 설명을 제공합니다.
- `schemas`는 모델 자체의 도구 호출에 사용하는 스키마입니다. 지금은 비워 두고 [내장 도구 호출 실습](chapter05_native_tool_calling.ipynb)에서 구현합니다.
- `prompt`는 도구를 어떤 형식으로 요청해야 하는지 모델에 알려 줍니다.

In [ ]:
from typing import Callable

class Tools:
    """에이전트가 사용할 도구를 등록하고 관리한다."""

    def __init__(self, requires_approval: list[str] = []):
        """도구 목록을 초기화하고 실행 전 승인이 필요한 도구를 지정한다.
        """
        self.registry = {}
        self.requires_approval = requires_approval

    def add_tool(self, name: str, func: Callable, description: str):
        """에이전트가 사용할 도구를 등록한다.

        매개변수:
            name: 도구의 이름.
            func: 도구의 동작을 구현한 함수.
            description: 도구의 용도.
        """
        self.registry[name] = {"function": func, "description": description}

    @property
    def schemas(self):
        """모델 자체의 도구 호출에서만 사용한다."""
        return None

    @property
    def descriptions(self):
        """등록한 모든 도구의 설명을 반환한다."""
        return "\n".join(
            f"`{tool}`: {self.registry[tool]['description']}" for tool in self.registry
        )

    @property
    def prompt(self):
        return f"""
# Tools

If needed, you can only use the following tools to assist you in completing tasks:

{self.descriptions}

To use a tool, respond with JSON: {{"tool": "name", "kwargs": {{"param": "value"}}}}
"""

함수와 설명을 함께 등록합니다.

In [ ]:
# 도구 등록
tools = Tools()
tools.add_tool("multiply", multiply, "Multiplies two numbers: multiply(a: str, b: str)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(location: str)")
print(tools.prompt)

인자가 많거나 동작이 복잡한 함수에는 각 인자의 자료형과 의미, 반환값의 형식까지 설명해야 합니다.

모델이 JSON 형식으로 답하도록 지시하면 `TinyAgent`가 도구 이름과 인자를 추출할 수 있습니다.

## 5. 도구 호출

모델이 출력할 수 있는 문자열의 예입니다.

```python
response = '{"tool": "calculator", "kwargs": {"a": "value_1", "b": "value_2"} }'
```

프로그램이 이 문자열에서 JSON을 읽어 실제 함수를 호출합니다. `Tools`에 다음 메서드를 추가합니다.

- `parse`는 응답에 도구 호출이 있는지 살펴보고 이름과 인자를 해석합니다.
- `execute`는 등록한 도구를 호출합니다.
- `observation`은 도구 결과를 대화 기록에 넣을 형식으로 바꿉니다.
- `is_done`은 도구 호출이 남아 있는지 확인합니다.

In [ ]:
import json
from typing import Any, Callable

from illustrated_agents.chapters.ch2 import Response


class Tools:
    """에이전트가 사용할 도구를 등록하고 관리한다."""

    def __init__(self, requires_approval: list[str] = []):
        """도구 목록을 초기화하고 실행 전 승인이 필요한 도구를 지정한다.
        """
        self.registry = {}
        self.requires_approval = requires_approval

    def add_tool(self, name: str, func: Callable, description: str = ""):
        """에이전트가 사용할 도구를 등록한다.

        매개변수:
            name: 도구의 이름.
            func: 도구의 동작을 구현한 함수.
            description: 도구의 용도.
        """
        self.registry[name] = {"function": func, "description": description}

    @property
    def descriptions(self) -> str:
        """등록한 모든 도구의 설명을 반환한다."""
        return "\n".join(
            f"`{tool}`: {self.registry[tool]['description']}" for tool in self.registry
        )

    @property
    def prompt(self) -> str:
        return f"""
# Tools

If needed, you can only use the following tools to assist you in completing tasks:

{self.descriptions}

To use a tool, respond with JSON: {{"tool": "name", "kwargs": {{"param": "value"}}}}
"""

    def parse(self, response: Response) -> Response:
        """텍스트에서 JSON 형식의 도구 호출을 추출한다."""
        text = response.content

        if '"tool":' in text or '"tool:"' in text:
            start, end = text.find("{"), text.rfind("}") + 1
            tool_call = json.loads(text[start:end])

            # 응답에 해석한 도구 호출 추가
            return Response(
                content=response.content,
                reasoning=response.reasoning,
                tool_call=tool_call,
            )

        return response

    def execute(self, response: Response) -> Any:
        """등록한 도구를 실행한다.

        매개변수:
            tool_call: "tool"과 "kwargs" 키를 담은 도구 호출 딕셔너리.
        """
        tool_call = response.tool_call
        name, kwargs = tool_call["tool"], tool_call.get("kwargs", {})

        # 승인이 필요한 도구는 실행 전에 사용자에게 확인
        if name in self.registry and name in self.requires_approval:
            response = input(f"Allow {name}? [y/N] ").strip().lower()
            if response not in ("y", "yes"):
                return f"Tool '{name}' was denied by the user."

        # 등록한 도구 처리
        if name in self.registry:
            tool_func = self.registry[name]["function"]
            return tool_func(**kwargs)

        return f"Tool '{name}' not found."

    def observation(self, result):
        """관찰 결과를 사용자 메시지로 반환한다."""
        return "user", f"OBSERVATION: {result}"

    def is_done(self, response: Response) -> bool:
        """도구 호출이 없거나 
        final_answer 도구를 사용하면 TinyAgent가 작업을 마친다."""
        if not response.tool_call:
            return True
        if response.tool_call["tool"] == "final_answer":
            response.content = response.tool_call.get("kwargs", "")
            return True
        return False

    @property
    def schemas(self):
        """모델 자체의 도구 호출에서만 사용한다."""
        return None

먼저 모델 응답에서 도구 호출을 읽는 코드를 살펴봅니다. 이 예제는 구조화된 JSON 출력을 보장하지 않는 모델의 텍스트를 간단하게 처리하는 방법입니다.

In [ ]:
from illustrated_agents.chapters.ch5 import parse_tool_annotated; parse_tool_annotated

중첩된 딕셔너리처럼 복잡한 JSON은 이 예제의 추출 방식으로 처리하기 어렵습니다. 이 경우 텍스트를 더 정교하게 처리하거나 모델 서버의 구조화 출력 기능을 사용할 수 있습니다.

In [ ]:
from illustrated_agents.chapters.ch5 import execute_tool_annotated; execute_tool_annotated

`observation`은 도구의 반환값과 이를 기록할 메시지 역할을 돌려줍니다. 프롬프트 방식에서는 `user` 역할을 쓰고, 내장 도구 호출에서는 `tool` 역할을 사용할 수 있습니다.

In [ ]:
from illustrated_agents.chapters.ch5 import observation_tool_annotated; observation_tool_annotated

`is_done`으로 도구 호출이 없다고 판정하면 에이전트가 `Response.content`를 최종 답변으로 반환합니다.

In [ ]:
from illustrated_agents.chapters.ch5 import done_tool_annotated; done_tool_annotated

In [ ]:
# 도구
tools = Tools()
tools.add_tool("multiply", multiply, "Multiplies two numbers: multiply(a: str, b: str)")

# 모델 응답 예시
response = Response(content='{"tool": "multiply", "kwargs": {"a": "3.1", "b": "6.5"} }')

# 도구 호출 해석
response = tools.parse(response)

# 도구 실행
tools.execute(response)

## 6. 도구 결과 처리와 `agent.py` 수정

`TinyAgent`에 도구 사용 지침과 실행 코드를 연결합니다.

- `system_prompt`에 사용할 수 있는 도구와 호출 형식을 넣습니다.
- `_execute_action`에서 모델이 요청한 도구를 실행합니다. 이 예제는 한 단계만 수행하므로 도구의 반환값을 사용자에게 바로 돌려줍니다.

In [ ]:
from illustrated_agents.chapters.ch2 import Trajectory
from illustrated_agents.chapters.ch4 import Memory


class TinyAgent:
    """교육용 에이전트의 기본 클래스를 정의한다."""

    def __init__(self, llm: LLM, memory: Memory, tools: Tools):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = None  # 6장에서 계획 기능 추가

        self.trajectory = Trajectory()

        # 시스템 프롬프트에 각 구성 요소의 지침을 추가
        system_prompt = "You are a helpful assistant.\n\n"
        system_prompt += self.tools.prompt
        self.memory.add("system", system_prompt)

    def run(self, task: str) -> str:
        """주어진 작업으로 에이전트를 실행한다."""
        self.memory.add("user", task)
        self.trajectory.initialize(task)

        return self._step()

    def _step(self) -> str:
        """에이전트의 한 단계를 실행한다."""
        # THOUGHT: 응답을 생성해 메모리에 추가
        response = self.llm.generate(self.memory.get_messages(), tools=self.tools.schemas)
        self.memory.add("assistant", response.content, tool_call=response.tool_call)

        # 도구 호출 해석
        response = self.tools.parse(response)

        # ANSWER: 종료 조건 확인
        if self.tools.is_done(response):
            self.trajectory.add(response)
            return response.content

        return self._execute_action(response)

    def _execute_action(self, response: Response) -> None:
        """도구가 수행할 동작을 실행한다."""

        # ACTION: 도구 실행
        result = self.tools.execute(response)

        # OBSERVATION: 도구 결과를 메모리에 추가하고 화면에 표시
        role, observation = self.tools.observation(result)
        self.memory.add(role, observation)
        self.trajectory.add(response, observation)

        return observation

먼저 새로 만든 `system_prompt`와 이전 `TinyAgent` 코드의 차이를 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch5 import agent_init_annotated; agent_init_annotated

In [ ]:
from illustrated_agents.utils import DiffViewer
from illustrated_agents.chapters import ch4, ch5
DiffViewer(ch4.TinyAgent.__init__, ch5.TinyAgent.__init__, "ch4.TinyAgent", "ch5.TinyAgent")

다음은 모델이 생성한 응답에 도구 호출이 있는지 검사하는 부분입니다. 이전 코드와 비교합니다.

In [ ]:
from illustrated_agents.chapters.ch5 import agent_step_annotated; agent_step_annotated

In [ ]:
DiffViewer(ch4.TinyAgent._step, ch5.TinyAgent._step, "ch4.TinyAgent", "ch5.TinyAgent")

`_execute_action`은 응답에서 도구 이름과 인자를 읽고, 함수를 실행한 뒤 결과를 메모리에 추가합니다. 이후 장에서는 이 메서드를 통해 스킬 지침을 읽거나 중간 결과를 전달합니다.

In [ ]:
from illustrated_agents.chapters.ch5 import agent_execute_action_annotated; agent_execute_action_annotated

In [ ]:
DiffViewer(ch4.TinyAgent._execute_action, ch5.TinyAgent._execute_action, "ch4.TinyAgent", "ch5.TinyAgent")

`agent.py`에서 바뀐 부분을 한 번에 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch5 import tinyagents_diff; tinyagents_diff

`Memory`와 `Tools`를 모두 전달해 `TinyAgent`를 생성합니다.

In [ ]:
# 도구
tools = Tools()
tools.add_tool("multiply", multiply, "Multiplies two numbers: multiply(a: str, b: str)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(location: str)")

# 메모리
memory = Memory()

# 에이전트 초기화
agent = TinyAgent(llm=llm, memory=memory, tools=tools)

도구가 필요한 질문을 보내 에이전트가 어떤 도구를 선택하는지 확인합니다.

예제 질문은 “5.1 곱하기 7.3은 얼마인가요?”입니다.

In [ ]:
agent.run("What is 5.1 times 7.3?")

저자의 예에서는 에이전트가 적절한 도구를 선택했습니다. 모델은 지시와 다른 출력을 생성할 수도 있습니다. 예상과 다르게 동작했다면 아래 실행 기록에서 모델의 응답과 도구 호출을 확인합니다.

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

도구 결과가 에이전트의 메모리에 어떻게 남았는지도 확인합니다.

In [ ]:
from rich import print
print(agent.memory.get_messages())

이번에는 도구가 필요 없는 질문을 보냅니다.

이번 질문은 “홍학에 대해 두 문장으로 알려 주세요”입니다.

In [ ]:
agent.run("Hi! Tell me something about flamingos in two sentences.")

에이전트는 직접 답할지 도구를 사용할지 선택합니다. 도구가 수백 개면 설명이 컨텍스트 창을 많이 차지하고 도구를 잘못 선택할 가능성도 있습니다. 실행 기록에서 이번 질문에 어떤 선택을 했는지 확인합니다.

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

---

# 구현한 내용

도구를 정의하고 모델의 요청을 해석해 함수를 실행했습니다. `tools.py`에는 도구를 관리하는 클래스를, `toolbox.py`에는 실제 도구 함수를 둡니다.

In [ ]:
from illustrated_agents.chapters.ch5 import what_we_built; what_we_built

# 다음 실습

1. [MCP](chapter05_mcp.ipynb)에서는 프롬프트 방식으로 도구를 호출하는 모델을 MCP 서버와 연결합니다.
2. [내장 도구 호출](chapter05_native_tool_calling.ipynb)에서는 Gemma 4 E4B가 도구를 요청하는 형식을 살펴봅니다.
3. [스킬](chapter05_skills.ipynb)에서는 에이전트가 작업에 필요한 지침을 읽게 합니다.

[6장](../chapter06/chapter06.ipynb)에서는 `for` 반복문으로 추론과 행동을 반복하는 ReAct를 구현합니다.